# DPRSC — C1/C3 full-scale accuracy across all 3 datasets

Reproduces **C1** (efficient algorithms, small additive error) and **C3** (significantly outperform baselines in accuracy) of *Differentially Private Range Subgraph Counting* (ICML 2026, `QYpByrxSTg`) at the paper's **full dataset coverage**: **ca-netscience (n=379)**, **musae-squirrel (n=5201)**, **bio-WormNet-v3 (n=16347)**, all patterns (edge, 2-star, triangle).

For each (dataset, pattern): run the ε-test and compare **ours** (`pure_DP`, `approx_DP`) vs **baselines** (`base_comp` basic composition, `base_comp_ADP` advanced composition). C3 holds when ours' error ≤ both baselines at every ε.

**Pure Python — a GPU does NOT help** (CPU/RAM-bound). To fit Colab's CPU/RAM, the large datasets use a reduced query count (`qmult`); the accuracy *ordering* (ours ≪ baselines) is structural so this does not change the verdict, only tightens the estimate. `workers=1–2` controls memory; each pattern is wrapped in try/except so a heavy `triangle` OOM still leaves `edge`+`2-star` for that dataset.

In [ ]:
# Clone official repo (code + bundled datasets) + install deps
!pip -q install numpy pandas matplotlib
![ ! -d DPRSC ] && git clone --depth 1 https://github.com/Airleave/DPRSC
import os, sys
os.chdir('/content/DPRSC'); sys.path.insert(0, '/content/DPRSC')
print('ready at', os.getcwd(), '| cpus:', os.cpu_count())

In [ ]:
# Helper: epsilon (accuracy) test for one (dataset, pattern) at d=1.
import math, logging, concurrent.futures
import numpy as np, pandas as pd
logging.basicConfig(level=logging.WARNING); logger = logging.getLogger('c1c3')
import preprocessing, ourAlg, baseline

def eps_run_acc(dataset, n, pattern, eps_list, workers=2, qmult=1.0):
    d = 1
    d_max, test_nodes, edges, h, m = preprocessing.graph_data_load(dataset, pattern, n, logger, d)
    Qn = math.ceil(n**1.5 * qmult); Q = preprocessing.generate_queries(Qn, m, d)
    rows = []
    with concurrent.futures.ProcessPoolExecutor(max_workers=workers) as ex:
        ftrue = ex.submit(ourAlg.query_true, n, m, d, Q, test_nodes, logger)
        fp = {e: ex.submit(ourAlg.pure_DP, n, m, d, Q, e, test_nodes, pattern, logger) for e in eps_list}
        fa = {e: ex.submit(ourAlg.approx_DP, n, m, d, Q, d_max, e, 1e-5, test_nodes, pattern, logger) for e in eps_list}
        true = ftrue.result()
        for e in eps_list:
            pe, _ = fp[e].result(); ae, _ = fa[e].result()
            bc  = baseline.base_comp(n, Qn, 1, e, true, pattern, logger)
            bcA = baseline.base_comp_ADP(n, Qn, 1, d_max, e, 1e-5, true, pattern, logger)
            rows.append(dict(eps=e, pure=pe/Qn, approx=ae/Qn, base_comp=bc[0], base_comp_ADP=bcA[0]))
    return pd.DataFrame(rows), m, Qn
print('helper ready')

In [ ]:
# CONFIG: all 3 datasets x all patterns. qmult shrinks Q on the big datasets (time/RAM).
EPS = [0.5, 1.0, 2.0, 4.0]
W = min(os.cpu_count() or 2, 2)   # workers=1-2 -> memory safe (triangle enumerations are heavy)
CONFIG = [
    ('ca-netscience',   379,   1.00, ['edge', '2star', 'triangle']),  # tiny -> full Q
    ('musae-squirrel',  5201,  0.10, ['edge', '2star', 'triangle']),  # 10x fewer queries
    ('bio-WormNet-v3',  16347, 0.02, ['edge', '2star', 'triangle']),  # 50x fewer queries
]
all_results = {}
for ds, n, qm, pats in CONFIG:
    for pat in pats:   # light -> heavy, so partial progress if triangle OOMs
        key = (ds, pat)
        try:
            print(f'>> {ds} {pat} (n={n}, qmult={qm}, workers={W}) ...', flush=True)
            df, m, Qn = eps_run_acc(ds, n, pat, EPS, workers=W, qmult=qm)
            all_results[key] = (df, m, Qn)
            print(df.round(3).to_string(index=False), flush=True)
        except Exception as e:
            print(f'  !! {ds} {pat} FAILED: {type(e).__name__}: {str(e)[:120]}', flush=True)
            all_results[key] = None
print('DONE')

In [ ]:
# C1/C3 summary: per (dataset, pattern), does ours beat both baselines at every eps?
rows = []
for (ds, pat), res in all_results.items():
    if res is None:
        rows.append(dict(dataset=ds, pattern=pat, status='FAILED')); continue
    df, m, Qn = res
    pure_ok   = (df.pure <= df.base_comp).all() and (df.pure <= df.base_comp_ADP).all()
    approx_ok = (df.approx <= df.base_comp).all() and (df.approx <= df.base_comp_ADP).all()
    r = df.iloc[1]  # eps = 1.0
    rows.append(dict(dataset=ds, pattern=pat, Q=Qn,
                     pure_eps1=round(r.pure,4), base_comp_eps1=round(r.base_comp,4),
                     pure_wins_all_eps=bool(pure_ok), approx_wins_all_eps=bool(approx_ok)))
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
n_ok = summary.get('pure_wins_all_eps'); 
if n_ok is not None:
    print(f"\nC3 (pure_DP <= both baselines at all eps): {int(n_ok.sum())}/{len(n_ok)} (dataset,pattern) cells")
summary.to_csv('c1c3_summary.csv', index=False)

In [ ]:
# Plot: mean error ours vs baselines per (dataset, pattern) that succeeded (log scale).
import matplotlib.pyplot as plt
ok = {k: v for k, v in all_results.items() if v is not None}
keys = list(ok.keys())
if keys:
    n_plots = len(keys); cols = 3; rows_n = (n_plots + cols - 1)//cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(5*cols, 3.5*rows_n), squeeze=False)
    methods = ['pure', 'approx', 'base_comp', 'base_comp_ADP']
    for ax, k in zip(axes.flat, keys):
        df, m, Qn = ok[k]
        means = [df[x].mean() for x in methods]
        ax.bar(methods, means, color=['#2ca02c','#98df8a','#d62728','#ff9896'])
        ax.set_yscale('log'); ax.set_title(f'{k[0]} · {k[1]}', fontsize=9)
        ax.tick_params(axis='x', labelsize=7, rotation=15)
    for ax in axes.flat[n_plots:]: ax.axis('off')
    plt.tight_layout(); plt.savefig('c1c3_fullscale.png', dpi=110, bbox_inches='tight'); plt.show()
    print('saved c1c3_fullscale.png')
else:
    print('no successful runs to plot')

In [ ]:
# Save all per-(dataset,pattern) CSVs + summary + figure, zip, download.
import glob, zipfile
from google.colab import files
for (ds, pat), res in all_results.items():
    if res is None: continue
    df, m, Qn = res
    df.assign(dataset=ds, pattern=pat, m=m, Q=Qn).to_csv(f'c1c3_{ds}_{pat}.csv', index=False)
with zipfile.ZipFile('/content/dprsc_c1c3_results.zip', 'w') as zf:
    for f in glob.glob('c1c3_*'):
        zf.write(f)
files.download('/content/dprsc_c1c3_results.zip')
print('zipped', glob.glob('c1c3_*'))